# 01 — Exploratory Data Analysis
AQI forecasting for Lahore. Run this AFTER you've backfilled at least a few months of data
(`python src/feature_pipeline.py backfill 365`).

Covers: univariate, bivariate, multivariate, and time-series-specific analysis
(ACF/PACF, seasonal decomposition, ADF stationarity test).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.utils.hopsworks_utils import get_feature_store
from src import config

sns.set_theme(style="whitegrid")
%matplotlib inline

## Load data from the Hopsworks Feature Store

In [ ]:
fs = get_feature_store()
fg = fs.get_feature_group(name=config.FEATURE_GROUP_NAME, version=config.FEATURE_GROUP_VERSION)
df = fg.read()
df = df.sort_values('timestamp').reset_index(drop=True)

# Cache a local snapshot so the Streamlit EDA page doesn't need a live Hopsworks call
os.makedirs('../data', exist_ok=True)
df.to_parquet('../data/eda_snapshot.parquet')

print(df.shape)
df.head()

## 7 questions about the data (structure check)

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nDuplicate timestamps:', df['timestamp'].duplicated().sum())
df.describe().T

## Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['aqi'], kde=True, ax=axes[0])
axes[0].set_title('AQI Distribution')
sns.boxplot(x=df['aqi'], ax=axes[1])
axes[1].set_title('AQI Boxplot (outlier check)')
plt.tight_layout(); plt.show()

In [ ]:
pollutants = [c for c in ['pm2_5', 'pm10', 'co', 'no2', 'o3', 'so2', 'nh3'] if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, pollutants):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(df['timestamp'], df['aqi'])
plt.title('AQI Over Time — Lahore')
plt.xlabel('Date'); plt.ylabel('AQI')
plt.show()

## Bivariate analysis — AQI vs weather variables

In [ ]:
weather_cols = [c for c in ['temperature', 'humidity', 'wind_speed', 'pressure'] if c in df.columns]
fig, axes = plt.subplots(1, len(weather_cols), figsize=(18, 4))
for ax, col in zip(axes, weather_cols):
    sns.scatterplot(x=df[col], y=df['aqi'], alpha=0.3, ax=ax)
    ax.set_title(f'AQI vs {col}')
plt.tight_layout(); plt.show()

**Note (Lahore-specific):** expect a negative relationship between `wind_speed` and AQI
(higher wind disperses pollution) and a positive relationship between `humidity` and AQI
during smog season (secondary particle formation). If these patterns don't show up,
revisit whether the weather-interaction features are being computed correctly.

## Multivariate analysis — correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(16, 12))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap — All Features')
plt.show()

# Top correlated features with the 72h-ahead target — useful for feature selection
if 'aqi_target_72h' in numeric_df.columns:
    print(numeric_df.corr()['aqi_target_72h'].sort_values(ascending=False).head(15))

## Time-series-specific: stationarity, ACF/PACF, seasonal decomposition
These directly justify the lag-feature and modeling choices made in `feature_engineering.py`.

In [ ]:
# Augmented Dickey-Fuller test — is the AQI series stationary?
result = adfuller(df['aqi'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print('=> Stationary (reject H0)' if result[1] < 0.05 else '=> Non-stationary (fail to reject H0) — consider differencing')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df['aqi'].dropna(), lags=72, ax=axes[0])
plot_pacf(df['aqi'].dropna(), lags=72, ax=axes[1])
plt.tight_layout(); plt.show()
# Significant spikes here justify WHICH lag features to keep (e.g. strong spike
# at lag 24 => daily cycle => lag_24h and lag_168h are worth keeping).

In [ ]:
# Seasonal decomposition — needs a DatetimeIndex and a regular hourly frequency
ts = df.set_index('timestamp')['aqi'].asfreq('h').interpolate()
decomposition = seasonal_decompose(ts, model='additive', period=24)  # daily cycle
fig = decomposition.plot()
fig.set_size_inches(14, 8)
plt.tight_layout(); plt.show()

## Smog season vs. normal season — does AQI actually differ?
Confirms the premise behind stratified evaluation in `training_pipeline.py` (`is_smog_season` feature, added in `feature_engineering.py`).

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['is_smog_season'].map({0: 'Normal', 1: 'Smog (Oct-Jan)'}), y=df['aqi'])
plt.title('AQI: Smog Season vs. Normal Season')
plt.xlabel(''); plt.ylabel('AQI')
plt.show()

print(df.groupby('is_smog_season')['aqi'].describe())

## Takeaways to carry into feature engineering / modeling

- (Fill in after running the cells above, e.g.:)
- AQI shows a clear daily cycle → cyclical hour features + lag_24h justified
- ADF test result: ... → stationary / needs differencing
- Wind speed has a [positive/negative] relationship with AQI, as expected for Lahore
- Strongest correlated features with the 72h target: ...
